# 04 · KPI Dashboard & Business Recommendations

**Goal:** Consolidate all findings into a single-page KPI summary and produce 3–5 actionable business recommendations.

---
This notebook is your **portfolio presentation layer** — keep it clean, business-focused, and free of raw code clutter.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mtick
import seaborn as sns

from src.visualizations import plot_heatmap

%matplotlib inline
sns.set_theme(style='whitegrid')
pd.set_option('display.float_format', '{:.2f}'.format)

df = pd.read_csv('../data/superstore_clean.csv', parse_dates=['Order Date', 'Ship Date'])
print(f'Loaded: {df.shape}')

## 1. KPI Summary Table

In [ ]:
kpis = {
    'Total Revenue':       f"${df['Sales'].sum():,.0f}",
    'Total Profit':        f"${df['Profit'].sum():,.0f}",
    'Overall Margin':      f"{df['Profit'].sum() / df['Sales'].sum() * 100:.1f}%",
    'Total Orders':        f"{df['Order ID'].nunique():,}",
    'Total Customers':     f"{df['Customer ID'].nunique():,}",
    'Avg Order Value':     f"${df.groupby('Order ID')['Sales'].sum().mean():,.2f}",
    'Avg Discount':        f"{df['Discount'].mean():.1%}",
    'Avg Ship Lag (days)': f"{df['Ship Lag (days)'].mean():.1f}",
    'Date Range':          f"{df['Order Date'].min().date()} → {df['Order Date'].max().date()}",
}

kpi_df = pd.DataFrame(list(kpis.items()), columns=['KPI', 'Value'])
kpi_df.style.set_properties(**{'text-align': 'left'}).hide(axis='index')

## 2. One-Page Dashboard

In [ ]:
import calendar

fig = plt.figure(figsize=(18, 14))
fig.suptitle('Superstore Sales — Executive Dashboard', fontsize=18, fontweight='bold', y=0.98)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

PALETTE = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

# ── Panel 1: Sales by Category (top-left) ───────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
cat_sales = df.groupby('Category')['Sales'].sum().sort_values()
ax1.barh(cat_sales.index, cat_sales.values, color=PALETTE[:len(cat_sales)])
ax1.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax1.set_title('Sales by Category')

# ── Panel 2: Profit by Category (top-middle) ────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
cat_profit = df.groupby('Category')['Profit'].sum().sort_values()
colors_p = ['#C44E52' if v < 0 else '#55A868' for v in cat_profit.values]
ax2.barh(cat_profit.index, cat_profit.values, color=colors_p)
ax2.axvline(0, color='black', linewidth=0.8)
ax2.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax2.set_title('Profit by Category')

# ── Panel 3: Sales by Segment (top-right) ───────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
seg = df.groupby('Segment')['Sales'].sum()
ax3.pie(seg.values, labels=seg.index, autopct='%1.1f%%',
        colors=PALETTE[:len(seg)], startangle=90)
ax3.set_title('Sales by Segment')

# ── Panel 4: Sales by Region (middle-left) ──────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
reg = df.groupby('Region')[['Sales', 'Profit']].sum().sort_values('Sales', ascending=False)
x = range(len(reg))
ax4.bar([i - 0.2 for i in x], reg['Sales'], 0.4, label='Sales', color=PALETTE[0])
ax4.bar([i + 0.2 for i in x], reg['Profit'], 0.4, label='Profit', color=PALETTE[2])
ax4.set_xticks(x)
ax4.set_xticklabels(reg.index, fontsize=8)
ax4.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax4.set_title('Sales & Profit by Region')
ax4.legend(fontsize=8)

# ── Panel 5: Monthly trend (middle, spans 2 cols) ───────────────────────────
ax5 = fig.add_subplot(gs[1, 1:])
monthly = df.groupby('Order YearMonth')['Sales'].sum().reset_index().sort_values('Order YearMonth')
ax5.plot(monthly['Order YearMonth'], monthly['Sales'], marker='o', linewidth=1.5,
         color=PALETTE[0], markersize=3)
ax5.fill_between(range(len(monthly)), monthly['Sales'], alpha=0.1, color=PALETTE[0])
ax5.set_xticks(range(0, len(monthly), 3))
ax5.set_xticklabels(monthly['Order YearMonth'].iloc[::3], rotation=45, ha='right', fontsize=7)
ax5.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax5.set_title('Monthly Sales Trend')

# ── Panel 6: Top 5 sub-categories by profit (bottom-left) ───────────────────
ax6 = fig.add_subplot(gs[2, 0])
top5 = df.groupby('Sub-Category')['Profit'].sum().sort_values(ascending=False).head(5)
ax6.barh(top5.index, top5.values, color=PALETTE[2])
ax6.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax6.set_title('Top 5 Sub-Categories by Profit')

# ── Panel 7: Bottom 5 sub-categories by profit (bottom-middle) ──────────────
ax7 = fig.add_subplot(gs[2, 1])
bot5 = df.groupby('Sub-Category')['Profit'].sum().sort_values().head(5)
ax7.barh(bot5.index, bot5.values, color=PALETTE[3])
ax7.axvline(0, color='black', linewidth=0.8)
ax7.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax7.set_title('Bottom 5 Sub-Categories (Loss)')

# ── Panel 8: Avg margin by discount bucket (bottom-right) ───────────────────
ax8 = fig.add_subplot(gs[2, 2])
df['Discount Bucket'] = pd.cut(df['Discount'],
                                bins=[-0.01, 0, 0.1, 0.2, 0.3, 0.5, 1.01],
                                labels=['0%', '1–10%', '11–20%', '21–30%', '31–50%', '>50%'])
disc_margin = df.groupby('Discount Bucket', observed=True)['Profit Margin (%)'].mean()
colors_d = ['#55A868' if v >= 0 else '#C44E52' for v in disc_margin.values]
ax8.bar(disc_margin.index, disc_margin.values, color=colors_d)
ax8.axhline(0, color='black', linewidth=0.8)
ax8.yaxis.set_major_formatter(mtick.PercentFormatter())
ax8.set_title('Avg Profit Margin by Discount Level')
ax8.set_xlabel('Discount Range')

plt.savefig('../data/dashboard.png', bbox_inches='tight', dpi=150)
print('Dashboard saved → data/dashboard.png')
plt.show()

## 3. Business Recommendations

Based on the analysis, fill in your findings below:

---

### 🔴 Issue 1: Loss-making sub-categories
**Finding:** *[e.g. Tables and Bookcases have negative total profit despite high sales volume.]*  
**Recommendation:** *[e.g. Review pricing strategy and supplier costs for these sub-categories, or bundle with high-margin items.]*

---

### 🔴 Issue 2: Excessive discounting
**Finding:** *[e.g. Discounts above 20% are strongly correlated with negative profit margins.]*  
**Recommendation:** *[e.g. Cap discounts at 15% except for strategic bulk orders; introduce minimum margin guardrails.]*

---

### 🟡 Opportunity 1: Region-specific growth
**Finding:** *[e.g. The South region shows the lowest profit margin despite comparable sales.]*  
**Recommendation:** *[e.g. Investigate product mix and shipping costs in Southern states; consider regional promotions for high-margin Technology products.]*

---

### 🟢 Opportunity 2: Technology category
**Finding:** *[e.g. Technology has the highest profit margins and is growing YoY.]*  
**Recommendation:** *[e.g. Increase marketing spend on Technology for Corporate and Home Office segments.]*

---

### 🟢 Opportunity 3: Seasonal peaks
**Finding:** *[e.g. Q4 consistently shows the highest sales — Q1 is weakest.]*  
**Recommendation:** *[e.g. Pre-position inventory before Q4; run demand-generation campaigns in Q1 to smooth revenue.]*